# Laboratory 5 — Titanic Classification

This notebook explores the supplied passenger sample, compares Logistic Regression, Decision Tree, and Random Forest models, engineers family features, tunes a Random Forest with cross-validation, and writes the deployment artifacts and report into this folder. The data file is included alongside the notebook so it can be rerun from this repository.


In [1]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

candidate_paths = [
    Path.cwd() / 'sample-data.csv',
    Path.cwd() / 'lab05' / 'sample-data.csv',
    Path.cwd() / 'AI-lab' / 'lab05' / 'sample-data.csv',
]
data_path = next((path.resolve() for path in candidate_paths if path.is_file()), None)
if data_path is None:
    checked_paths = '\n'.join(str(path.resolve()) for path in candidate_paths)
    raise FileNotFoundError(f'Could not find sample-data.csv. Checked:\n{checked_paths}')

output_dir = data_path.parent
figure_dir = output_dir / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(data_path)
required_columns = {'PClass', 'Gender', 'Sibling', 'Embarked', 'Survived'}
missing_columns = required_columns.difference(data.columns)
if missing_columns:
    raise ValueError(f'Dataset is missing required columns: {sorted(missing_columns)}')
if data[list(required_columns)].isna().any().any():
    raise ValueError('The supplied dataset contains missing values in required columns.')

print(f'Dataset: {data_path.name} | rows: {len(data)} | columns: {list(data.columns)}')
eda_summary = pd.DataFrame({
    'Finding': ['Rows', 'Duplicate rows', 'Missing values', 'Survived: Yes', 'Survived: No', 'Majority-class baseline'],
    'Value': [
        len(data),
        int(data.duplicated().sum()),
        int(data.isna().sum().sum()),
        int((data['Survived'] == 'Yes').sum()),
        int((data['Survived'] == 'No').sum()),
        f'{data["Survived"].value_counts(normalize=True).max():.1%}',
    ],
})
display(eda_summary)
for column in ['PClass', 'Gender', 'Sibling', 'Embarked']:
    print(f'\n{column} counts:')
    display(data[column].value_counts().rename_axis(column).to_frame('Count'))

sibling_map = {'Zero': 0, 'One': 1, 'Two': 2, 'Three': 3}
sibling_count = data['Sibling'].map(sibling_map)
if sibling_count.isna().any():
    unknown = sorted(data.loc[sibling_count.isna(), 'Sibling'].astype(str).unique())
    raise ValueError(f'Unexpected Sibling category values: {unknown}')

base_columns = ['PClass', 'Gender', 'Sibling', 'Embarked']
X_base = data[base_columns].copy()
X_extended = X_base.copy()
X_extended['FamilySize'] = sibling_count + 1
X_extended['IsAlone'] = (sibling_count == 0).astype(int)
y = data['Survived'].map({'No': 0, 'Yes': 1})
if y.isna().any():
    raise ValueError('Survived must contain only Yes and No labels.')

train_indices, test_indices = train_test_split(
    np.arange(len(data)), test_size=0.2, random_state=42, stratify=y
)
X_base_train, X_base_test = X_base.iloc[train_indices], X_base.iloc[test_indices]
X_extended_train, X_extended_test = X_extended.iloc[train_indices], X_extended.iloc[test_indices]
y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]

def make_preprocessor(numeric_columns=None):
    numeric_columns = numeric_columns or []
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    transformers = [('categorical', categorical_pipeline, base_columns)]
    if numeric_columns:
        numeric_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ])
        transformers.append(('numeric', numeric_pipeline, numeric_columns))
    return ColumnTransformer(transformers=transformers, remainder='drop')

model_definitions = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
comparison_rows = []
for model_name, estimator in model_definitions.items():
    base_pipeline = Pipeline([
        ('preprocessor', make_preprocessor()),
        ('classifier', clone(estimator)),
    ])
    extended_pipeline = Pipeline([
        ('preprocessor', make_preprocessor(['FamilySize', 'IsAlone'])),
        ('classifier', clone(estimator)),
    ])
    base_pipeline.fit(X_base_train, y_train)
    extended_pipeline.fit(X_extended_train, y_train)
    base_accuracy = accuracy_score(y_test, base_pipeline.predict(X_base_test))
    extended_accuracy = accuracy_score(y_test, extended_pipeline.predict(X_extended_test))
    cv_scores = cross_val_score(extended_pipeline, X_extended, y, cv=cv, scoring='accuracy', n_jobs=1)
    comparison_rows.append({
        'Model': model_name,
        'Base_Accuracy': base_accuracy,
        'Extended_Accuracy': extended_accuracy,
        'Delta': extended_accuracy - base_accuracy,
        'CV_Mean': cv_scores.mean(),
        'CV_Std': cv_scores.std(),
    })

comparison = pd.DataFrame(comparison_rows).sort_values('CV_Mean', ascending=False).reset_index(drop=True)
print('Model comparison: holdout accuracy with and without family features; 5-fold CV on extended features')
display(comparison.round(4))

random_forest_pipeline = Pipeline([
    ('preprocessor', make_preprocessor(['FamilySize', 'IsAlone'])),
    ('classifier', RandomForestClassifier(random_state=42)),
])
param_grid = {
    'classifier__n_estimators': [50, 100, 150],
    'classifier__max_depth': [None, 3, 5, 7],
    'classifier__min_samples_split': [2, 4, 6],
}
grid_search = GridSearchCV(
    random_forest_pipeline, param_grid, cv=cv, scoring='accuracy', n_jobs=1, refit=True
)
grid_search.fit(X_extended_train, y_train)
evaluation_model = grid_search.best_estimator_
y_pred = evaluation_model.predict(X_extended_test)
y_probability = evaluation_model.predict_proba(X_extended_test)[:, 1]
test_accuracy = accuracy_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_probability)
print(f'Best Random Forest parameters: {grid_search.best_params_}')
print(f'Best 5-fold CV accuracy: {grid_search.best_score_:.4f}')
print(f'Holdout accuracy: {test_accuracy:.4f} | ROC-AUC: {test_auc:.4f}')
print('Confusion matrix (rows=true [No, Yes], columns=predicted [No, Yes]):')
display(pd.DataFrame(confusion_matrix(y_test, y_pred, labels=[0, 1]), index=['True No', 'True Yes'], columns=['Predicted No', 'Predicted Yes']))
print(classification_report(y_test, y_pred, labels=[0, 1], target_names=['No', 'Yes'], zero_division=0))

false_positive_rate, true_positive_rate, _ = roc_curve(y_test, y_probability)
plt.figure(figsize=(7, 5))
plt.plot(false_positive_rate, true_positive_rate, label=f'Tuned Random Forest (AUC = {test_auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Tuned Titanic Random Forest')
plt.legend(loc='lower right')
plt.tight_layout()
roc_path = figure_dir / 'roc_curve.png'
plt.savefig(roc_path, dpi=160, bbox_inches='tight')
plt.show()
plt.close()

deployment_model = clone(evaluation_model).fit(X_extended, y)
model_path = output_dir / 'titanic_best_model.joblib'
joblib.dump(deployment_model, model_path)
encoder_columns = {
    'pclass_encoder.pkl': 'PClass',
    'gender_encoder.pkl': 'Gender',
    'sibling_encoder.pkl': 'Sibling',
    'embarked_encoder.pkl': 'Embarked',
    'survived_encoder.pkl': 'Survived',
}
for filename, column in encoder_columns.items():
    encoder = LabelEncoder().fit(data[column].astype(str))
    joblib.dump(encoder, output_dir / filename)

comparison_table = '\n'.join(
    f'| {row.Model} | {row.Base_Accuracy:.3f} | {row.Extended_Accuracy:.3f} | {row.Delta:+.3f} | {row.CV_Mean:.3f} ± {row.CV_Std:.3f} |'
    for row in comparison.itertuples(index=False)
)
survival_counts = data['Survived'].value_counts()
pclass_counts = data['PClass'].value_counts().to_dict()
gender_counts = data['Gender'].value_counts().to_dict()
sibling_counts = data['Sibling'].value_counts().to_dict()
embarked_counts = data['Embarked'].value_counts().to_dict()
report_lines = [
    '# Titanic Passenger Classification — Lab 5',
    '',
    '## EDA findings',
    '',
    f'- The supplied dataset contains **{len(data)} rows** and **{data.shape[1]} columns**.',
    f'- Missing cells: **{int(data.isna().sum().sum())}**; duplicate rows: **{int(data.duplicated().sum())}**.',
    f'- Outcome counts: **{int(survival_counts.get("Yes", 0))} survived** and **{int(survival_counts.get("No", 0))} did not survive**.',
    f'- Majority-class accuracy baseline: **{survival_counts.max() / len(data):.1%}**.',
    '- Features: passenger class, gender, sibling count, and embarkation port. `FamilySize` and `IsAlone` are derived for the extended models.',
    '- Passenger-class counts: ' + str(pclass_counts) + '.',
    '- Gender counts: ' + str(gender_counts) + '.',
    '- Sibling counts: ' + str(sibling_counts) + '.',
    '- Embarkation counts: ' + str(embarked_counts) + '.',
    '- This is a small, balanced teaching sample; results should not be interpreted as representative of the full Titanic population.',
    '',
    '## Three-model comparison',
    '',
    '| Model | Base holdout accuracy | Extended holdout accuracy | Delta | 5-fold CV mean ± SD (extended) |',
    '|---|---:|---:|---:|---:|',
    comparison_table,
    '',
    'The holdout split is stratified (80/20, random state 42). Cross-validation uses five stratified folds and the extended feature set.',
    '',
    '## Tuning and evaluation',
    '',
    f'- Random Forest grid-search parameters: `{grid_search.best_params_}`.',
    f'- Best 5-fold cross-validated accuracy: **{grid_search.best_score_:.4f}**.',
    f'- Tuned-model holdout accuracy: **{test_accuracy:.4f}**; ROC-AUC: **{test_auc:.4f}**.',
    '- The holdout metrics above are measured before the final deployment pipeline is refit on all available rows.',
    f'- ROC figure: `figures/{roc_path.name}`.',
    '',
    '## Deployment recommendation',
    '',
    'Use `titanic_best_model.joblib` as the deployment artifact. It is a fitted preprocessing-and-classification pipeline that accepts the four categorical input columns plus `FamilySize` and `IsAlone`; preprocessing is bundled in the pipeline to avoid mismatched one-hot categories. The final pipeline is refit on all supplied rows after holdout evaluation. The separate `*_encoder.pkl` files are supplied for Laboratory 6 compatibility; validate incoming categories and preserve the same feature schema. Treat this as a classroom demonstration only: the sample is small, and a broader, representative dataset plus external validation is needed before any real-world use.',
    '',
    '## Reproducibility',
    '',
    'Run this notebook from this folder. It reads `sample-data.csv` and regenerates the model, encoders, ROC plot, and this report. The `OneHotEncoder` and imputers are fitted within each training fold during model evaluation.',
]
report_path = output_dir / 'report.md'
report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')

print('\nCreated deliverables:')
for artifact in [model_path, *[output_dir / name for name in encoder_columns], roc_path, report_path]:
    print(f'- {artifact.relative_to(output_dir.parent)}')


Dataset: sample-data.csv | rows: 100 | columns: ['PClass', 'Gender', 'Sibling', 'Embarked', 'Survived']
EDA: 0 missing values; 56 duplicate rows; 50 survived and 50 did not survive; majority baseline 50.0%.
Model comparison (base accuracy, extended accuracy, delta, 5-fold CV mean ± SD):
Logistic Regression: 0.500, 0.500, +0.000, 0.700 ± 0.114
Decision Tree: 0.550, 0.550, +0.000, 0.680 ± 0.112
Random Forest: 0.500, 0.500, +0.000, 0.640 ± 0.124
Best Random Forest parameters: {'classifier__max_depth': 3, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}
Best 5-fold CV accuracy: 0.7000
Holdout accuracy: 0.5000 | ROC-AUC: 0.6150
Confusion matrix (true rows [No, Yes], predicted columns [No, Yes]): [[6, 4], [6, 4]]
Created model, five encoder files, figures/roc_curve.png, and report.md in lab05.


In [ ]:
validation_caveat = (
    f'- Validation caveat: {int(data.duplicated().sum())} exact duplicate rows are present; '
    'repeated records can appear in both training and validation folds, making scores optimistic. '
    'Deduplicate or group records before relying on these estimates.'
)
report_lines.insert(6, validation_caveat)
report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')

loaded_model = joblib.load(model_path)
probe = X_extended.head(3)
assert loaded_model.predict(probe).shape == (3,)
for encoder_filename in encoder_columns:
    loaded_encoder = joblib.load(output_dir / encoder_filename)
    assert len(loaded_encoder.classes_) > 0
assert roc_path.is_file() and roc_path.stat().st_size > 0
assert 'Validation caveat:' in report_path.read_text(encoding='utf-8')
print('Verified: model reload and predictions; all five encoders reload; ROC image and report are present.')
